<a href="https://colab.research.google.com/github/Nithya1015/Data-science-AI-ML-internship/blob/main/15_Used_Car_Price_Prediction_Summer_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="left" style="background-color: #008080; padding: 20px 10px;">
<h3><b>IDEAS - Institute of Data Engineering, Analytics and Science Foundation</b></h3>
<p>Summer Internship Program 2026</p>
<hr style="width:100%;">
<h3><b>Project Title:</b> Used Car Price Prediction Using Regression Models</h3>
<h4>Project Notebook</h4>

<blockquote style="border-left: 4px solid #4285F4; padding-left: 15px;">
  <strong>Created by:</strong> Suprava Das<br>
  <strong>Designation:</strong> Associate Software Developer
</blockquote>
<hr style="width:100%;">
</div>

### Question 1: Import Packages and Load the Dataset (1 Mark)

Import `pandas` as `pd`. Download the dataset `used_cars.csv` from https://drive.google.com/drive/folders/1gieHICVDBbUKMZiSF4YRQMAJic-w50JM?usp=sharing. Load the dataset `used_cars.csv` into a DataFrame named `df`.
Create a copy of the DataFrame named `df1` to preserve the original data for future reference.

**Expected Output:** The code cell should execute without any errors.

In [2]:
# Write your answer here
import pandas as pd

# Loading the dataset and making a copy as required by the instructions
df = pd.read_csv('used_cars.csv')
df1 = df.copy()

### Question 2: Shuffle and Display the Dataset (1 Mark)

Shuffle the dataset `df1` using `.sample(frac=1, random_state=42)` to ensure the data is randomly distributed. Display the top 10 rows of the shuffled dataset.

**Expected Output:** A table showing the first 10 rows of the randomized DataFrame.

In [3]:
# Write your answer here
# Shuffling the dataset to distribute data randomly
df1 = df1.sample(frac=1, random_state=42).reset_index(drop=True)

# Displaying the top 10 rows for a quick look at the randomized data
print(df1.head(10))

           brand                 model  model_year       milage fuel_type  \
0          Lexus           IS 300 Base        2018   50,992 mi.  Gasoline   
1      Chevrolet           Impala Base        2004   64,500 mi.  Gasoline   
2            RAM              2500 SLT        2017   86,000 mi.    Diesel   
3  Mercedes-Benz       SL-Class SL 550        2013   24,933 mi.  Gasoline   
4           Ford    Shelby GT350R Base        2018   18,500 mi.  Gasoline   
5            GMC             Yukon SLT        2018   85,500 mi.  Gasoline   
6          Volvo  V60 Cross Country T5        2020   10,500 mi.  Gasoline   
7         Nissan           Titan XD SV        2020   10,001 mi.  Gasoline   
8            BMW                 330 i        2005   59,300 mi.  Gasoline   
9          Lexus           RX 330 Base        2006  110,250 mi.  Gasoline   

                                              engine  \
0      260.0HP 3.5L V6 Cylinder Engine Gasoline Fuel   
1      180.0HP 3.4L V6 Cylinder Engine G

### Question 3: Dataset Overview and Missing Values (2 Marks)

Print the total number of data points (rows) in the dataset `df1`. Then, find and print the number of null (missing) values present in each column.

**Expected Output:** The total row count and a Series showing the count of nulls per column.

In [4]:
# Write your answer here
# Printing basic stats about the data size and missing values
print(f"Total number of data points: {len(df1)}")
print("Missing values per column:")
print(df1.isnull().sum())

Total number of data points: 4009
Missing values per column:
brand             0
model             0
model_year        0
milage            0
fuel_type       170
engine            0
transmission      0
ext_col           0
int_col           0
accident        113
clean_title     596
price             0
dtype: int64


### Question 4: Standardize Missing or Invalid Strings (2 Marks)

For all columns with an `object` (string) data type in `df1`, perform the following replacements to standardize missing data:
- Replace the specific character `'–'` with the string `'Unknown'`.
- Replace the string `'nan'` with `'Unknown'`.
- Replace any entirely empty strings or strings consisting only of whitespace with `'Unknown'`.

**Expected Output:** The code cell should execute without errors, standardizing the text data.

In [5]:
# Write your answer here
# Standardizing missing text data in 'object' columns
for col in df1.select_dtypes(include=['object']).columns:
    df1[col] = df1[col].astype(str).replace(['–', 'nan', '', ' '], 'Unknown')

### Question 5: Feature Engineering - Car Age (1 Mark)

Create a new column named `car_age` in `df1` by calculating the age of each car. You can do this by subtracting the `model_year` from the current year (use `2026` as the current year).
After calculating the age, drop the original `model_year` column from the DataFrame.

**Expected Output:** The code cell should execute without errors.

In [6]:
# Write your answer here
# Calculating car_age using 2026 as the current year
df1['car_age'] = 2026 - df1['model_year']

# Removing the original model_year column
df1.drop('model_year', axis=1, inplace=True)


### Question 6: Format Numeric Columns (1 Mark)

Clean the following two columns so they can be used as numbers:
1. **milage:** Remove the string `' mi.'` and any commas (`,`), then convert the column to integers.
2. **price:** Remove currency symbols (`$`) and any commas (`,`), then convert the column to floats.

**Expected Output:** The code cell should execute without errors, changing both columns to numeric types.

In [7]:
# Write your answer here
# Formatting 'milage' and 'price' into numeric types
df1['milage'] = df1['milage'].str.replace(' mi.', '').str.replace(',', '').astype(int)
df1['price'] = df1['price'].str.replace('$', '').str.replace(',', '').astype(float)

### Question 7: Handle Low-Frequency Categories (2 Marks)

Some categorical columns have too many rare values. For both the `transmission` and `ext_col` columns, group categories that appear in **less than 1%** of the total dataset into a new category called `'others'`.
Additionally, drop the `clean_title` column entirely as it provides little variance.

**Expected Output:** The code cell should execute without errors.

In [8]:
# Write your answer here
# Grouping rare categorical values and dropping 'clean_title'
for col in ['transmission', 'ext_col']:
    counts = df1[col].value_counts(normalize=True)
    df1[col] = df1[col].apply(lambda x: x if counts[x] >= 0.01 else 'others')

    # Removing column with little variance
df1.drop('clean_title', axis=1, inplace=True)

### Question 8: Remove Outliers and Apply Label Encoding (2 Marks)

Remove price outliers by filtering `df1` to keep only rows where the `price` is **below the 99th percentile**.
Next, import `LabelEncoder` from `sklearn.preprocessing` and apply it to convert all remaining categorical (`object`) columns into numerical values.

**Expected Output:** The code cell should execute without errors.

In [9]:
# Write your answer here
from sklearn.preprocessing import LabelEncoder

# Keeping only prices below the 99th percentile to remove outliers
q_99 = df1['price'].quantile(0.99)
df1 = df1[df1['price'] < q_99]

# Converting categorical text into numeric values
le = LabelEncoder()
for col in df1.select_dtypes(include=['object']).columns:
    df1[col] = le.fit_transform(df1[col])

### Question 9: Train-Test Split (1 Mark)

Define the `price` column as your target variable (`y`) and separate the remaining columns to form the feature set (`X`).
Split the features and target into training and testing datasets using an 80:20 ratio (`test_size=0.2`) and set `random_state=42`. Print the shapes of the resulting `X_train` and `X_test`.

**Expected Output:** A printout of the dimensions (shape) for the training and testing feature sets.

In [10]:
# Write your answer here
from sklearn.model_selection import train_test_split

# Defining features (X) and target (y)
X = df1.drop('price', axis=1)
y = df1['price']

# Performing an 80:20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

X_train shape: (3174, 10), X_test shape: (794, 10)


### Question 10: Model Training and Evaluation (4 Marks)

Import `RandomForestRegressor` from `sklearn.ensemble`, and `r2_score`, `mean_squared_error`, `mean_absolute_error` from `sklearn.metrics`.
Train the Random Forest model (with `n_estimators=100`, `random_state=42`) on the training data. Predict the prices for the test data and evaluate the model by printing the R² Score, MSE, and MAE.

**Expected Output:** Three numeric values representing the R² score, Mean Squared Error, and Mean Absolute Error of the model.

In [11]:
# Write your answer here
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Initializing and training the Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Making predictions and evaluating performance
y_pred = rf_model.predict(X_test)
print(f"R² Score: {r2_score(y_test, y_pred)}")
print(f"MSE: {mean_squared_error(y_test, y_pred)}")
print(f"MAE: {mean_absolute_error(y_test, y_pred)}")

R² Score: 0.7422473984607999
MSE: 357090522.2199794
MAE: 9981.921410579345


### Question 11: Load MNIST Data (3 Marks)

Load the MNIST dataset using `sklearn.datasets.load_digits`. Separate the dataset into features (`X`) and target labels (`y`).
Print the shape of the features and the target arrays.

**Expected Output:** The shape of `X` and `y`.

In [12]:
# Write your answer here
from sklearn.datasets import load_digits

# Loading the dataset and splitting into features and labels
digits = load_digits()
X_mnist, y_mnist = digits.data, digits.target

print(f"Features shape: {X_mnist.shape}, Target shape: {y_mnist.shape}")

Features shape: (1797, 64), Target shape: (1797,)


### Question 12: K-Means Clustering (5 Marks)

Import `KMeans` from `sklearn.cluster`. Initialize and fit the K-Means clustering model on the MNIST features (`X`).
Since we know there are 10 digits (0-9), set the number of clusters to 10. Assign the cluster labels to a variable `kmeans_labels`.

**Expected Output:** A successfully fitted K-Means model and the cluster labels array.

In [13]:
# Write your answer here
from sklearn.cluster import KMeans

# Fitting the model to find 10 digit groups
kmeans = KMeans(n_clusters=10, random_state=42)
kmeans_labels = kmeans.fit_predict(X_mnist)

### Question 13: F1 Score Evaluation for Clustering (5 Marks)

Evaluate the clustering performance using the F1 score. Since K-Means assigns arbitrary cluster labels, you will first need to map each cluster label to the most frequent true label in that cluster.
After mapping the labels, calculate and print the macro-averaged F1 score using `sklearn.metrics.f1_score`.

**Expected Output:** The calculated F1 score of the clustering.

In [14]:
# Write your answer here
import numpy as np
from scipy.stats import mode
from sklearn.metrics import f1_score

# Mapping cluster IDs to actual digit labels based on frequency
mapped_labels = np.zeros_like(kmeans_labels)
for i in range(10):
    mask = (kmeans_labels == i)
    mapped_labels[mask] = mode(y_mnist[mask], keepdims=True).mode

# Calculating the macro-averaged F1 score
f1 = f1_score(y_mnist, mapped_labels, average='macro')
print(f"Macro-averaged F1 Score: {f1}")

Macro-averaged F1 Score: 0.8627854786352394
